# Step 3 — Embeddings

Turning text into numbers that capture **meaning**.

Re-runs load + split from notebook 01 (fast, no API), then does the new part.

> **Quota:** embeddings have a **separate** quota from chat — this does not
> touch your 20 chat requests/day. But the free tier allows only
> **100 texts per minute**, which matters below.

## Setup — repeat steps 1 & 2

In [ ]:
import os, requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()
assert os.environ.get("GOOGLE_API_KEY"), "Put your key in rebuild/.env"

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; learning-project)"}

def load_url(url: str) -> Document:
    r = requests.get(url, headers=HEADERS, timeout=30); r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
        tag.decompose()
    main = soup.find("main") or soup.find("article") or soup.body
    return Document(
        page_content=main.get_text(separator="\n", strip=True),
        metadata={"source": url,
                  "title": soup.title.get_text(strip=True) if soup.title else url},
    )

URLS = [
    "https://en.wikipedia.org/wiki/Tata_Motors",
    "https://en.wikipedia.org/wiki/Mahindra_%26_Mahindra",
]
docs = [load_url(u) for u in URLS]

splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""], chunk_size=1000, chunk_overlap=200
)
chunks = splitter.split_documents(docs)
print(f"{len(docs)} documents -> {len(chunks)} chunks")

## 3.1 — What is an embedding?

An embedding turns text into a long list of numbers — a **vector**.

The useful property: **similar meaning produces similar vectors.** Not similar
*spelling* — similar *meaning*. That's what beats keyword search.

The tutorial used `OpenAIEmbeddings` (paid). Google's is free.

Note the model name: `text-embedding-004` from older tutorials **no longer
exists**. Current models are `gemini-embedding-001` and `gemini-embedding-2`.

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2")

vector = embeddings.embed_query("Who is the chairman of Tata Motors?")
print("dimensions:", len(vector))
print("first 8   :", [round(x, 4) for x in vector[:8]])

3072 numbers for one short question. No single dimension is
human-readable — the *pattern* across all of them carries the meaning.

## 3.2 — Proving it captures meaning

**Cosine similarity** measures the angle between two vectors: `1.0` = identical
direction, `0.0` = unrelated.

Watch two sentences that share almost no words but mean the same thing.

In [ ]:
import math

def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x*x for x in a)) * math.sqrt(sum(y*y for y in b)))

samples = [
    "Tata Motors reported record quarterly profit.",
    "The carmaker posted its highest ever earnings for the quarter.",  # same idea
    "I baked banana bread this morning.",                              # unrelated
]

vs = embeddings.embed_documents(samples)

print(f"same meaning, different words : {cosine(vs[0], vs[1]):.3f}")
print(f"completely unrelated          : {cosine(vs[0], vs[2]):.3f}")

Sentences 0 and 1 share only the word "the", yet score far higher than the
unrelated pair. **That gap is the whole basis of semantic search.**

Don't expect ~0.0 for unrelated text — these models place all English in roughly
the same region, so unrelated pairs still score 0.5-0.6. The **relative gap** is
what matters, not the absolute number.

## 3.3 — Embedding 108 chunks: the rate limit

The free tier allows **100 texts per minute**. We have 108, so a naive
`embed_documents(all_chunks)` fails with `429 RESOURCE_EXHAUSTED`.

Real systems handle this with **backoff**: catch the error, read how long the
API asks you to wait, sleep, retry. The helper below does that, and caches the
result to disk so re-running this notebook costs nothing.

In [ ]:
import time, re, numpy as np
from pathlib import Path

CACHE = Path("vectors_cache.npz")

def embed_all(texts, batch_size=40, verbose=True):
    """Embed a list of texts, backing off when the rate limit is hit."""
    out = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        while True:
            try:
                out.extend(embeddings.embed_documents(batch))
                if verbose:
                    print(f"  embedded {len(out)}/{len(texts)}")
                break
            except Exception as e:
                msg = str(e)
                if "RESOURCE_EXHAUSTED" not in msg and "429" not in msg:
                    raise                      # a real error, not throttling
                m = re.search(r"retry in ([\d.]+)s", msg)
                wait = float(m.group(1)) + 2 if m else 35
                if verbose:
                    print(f"  rate limit hit, waiting {wait:.0f}s...")
                time.sleep(wait)
    return out


chunk_texts = [c.page_content for c in chunks]

if CACHE.exists():
    chunk_vectors = np.load(CACHE)["v"].tolist()
    print(f"loaded {len(chunk_vectors)} vectors from cache (no API calls)")
else:
    t0 = time.time()
    chunk_vectors = embed_all(chunk_texts)
    np.savez_compressed(CACHE, v=np.array(chunk_vectors, dtype="float32"))
    print(f"\nembedded in {time.time()-t0:.0f}s and cached to {CACHE}")

print(f"{len(chunk_vectors)} vectors x {len(chunk_vectors[0])} dimensions")

> Expect **90-120 seconds** the first time, including two pauses for the rate
> limit. After that the cache makes it instant. Delete `vectors_cache.npz` to
> force a re-embed.

## 3.4 — Search by hand

Before FAISS does it for us, do it manually so you can see there's no magic:

1. Embed the question
2. Compare against all 108 chunk vectors
3. Return the closest

In [ ]:
def search(question, k=3):
    qv = embeddings.embed_query(question)
    scored = sorted(
        ((cosine(qv, cv), i) for i, cv in enumerate(chunk_vectors)),
        reverse=True,
    )
    print(f"Q: {question}\n" + "=" * 70)
    for score, i in scored[:k]:
        src = chunks[i].metadata["source"].split("/")[-1]
        snippet = chunks[i].page_content[:180].replace("\n", " ")
        print(f"[{score:.3f}] {src}\n   {snippet}...\n")

search("Which company makes the Mahindra Thar?")

In [ ]:
search("What vehicles does Tata Motors produce?")

Now try one phrased in words the articles do **not** use — that's the real
test of semantic search.

In [ ]:
search("How did the company perform financially?")

## Why we still need FAISS

`search()` compares against **every** chunk. Fine for 108, hopeless for 100,000
— minutes per query.

FAISS builds an index that finds nearest neighbours without checking everything,
and persists to disk so you never re-embed.

---

## Checkpoint

You have `chunk_vectors` — 108 vectors of 3072 numbers, cached to disk, and
proof that semantic search works.

**Next:** load them into FAISS and wire in Gemini for cited answers.